# Script per la creazione delle cartelle
Creo le cartelle `train/`, `test/` e `val/` contenenti il 70%, 15% e 15% delle coppie prese randomicamente

In [1]:
import os
import random
import shutil

In [4]:
def split_dataset(aligned_dir, output_dir,
                  train_ratio=0.70, val_ratio=0.15, test_ratio=0.15,
                  extension=".tif", seed=42):
    """
    Suddivide casualmente le coppie di immagini presenti in `aligned_dir`
    nelle cartelle train/val/test create in `output_dir`, con le proporzioni
    specificate da train_ratio, val_ratio e test_ratio.
    
    Parametri:
    - aligned_dir : cartella contenente tutte le immagini (coppie).
    - output_dir  : cartella dove verranno create train/, val/, test/.
    - train_ratio : percentuale di immagini per il training (default 70%).
    - val_ratio   : percentuale di immagini per la validation (default 15%).
    - test_ratio  : percentuale di immagini per il test (default 15%).
    - extension   : estensione dei file immagine (es. ".png", ".jpg").
    - seed        : seme per la generazione random (riproducibilità).
    """
    # Imposta il seed per ottenere sempre la stessa suddivisione, se necessario
    random.seed(seed)

    # Crea cartelle di destinazione (train/val/test)
    os.makedirs(os.path.join(output_dir, "train"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "val"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "test"), exist_ok=True)

    # 1) Individua i file che terminano con '_label_free' + extension
    all_files = os.listdir(aligned_dir)
    label_free_files = sorted([f for f in all_files if f.endswith("_label_free" + extension)])
    print(f"Totale file _label_free trovati: {len(label_free_files)}")

    # 2) Ricava il "prefisso" comune (es. "00000_08500") per ogni coppia
    #    e verifica che esista il corrispondente file "_stained"
    pairs_prefixes = []
    for lf_file in label_free_files:
        prefix = lf_file.replace("_label_free" + extension, "")
        stained_file = prefix + "_stained" + extension
        if stained_file in all_files:
            pairs_prefixes.append(prefix)
    print(f"Totale coppie trovate: {len(pairs_prefixes)}")
    print(f"Elenco primi 50 prefissi: {pairs_prefixes[:50]}")

    # 3) Mescola casualmente i prefissi per suddividere in train/val/test
    random.shuffle(pairs_prefixes)
    num_total = len(pairs_prefixes)

    train_end = int(num_total * train_ratio)
    val_end   = train_end + int(num_total * val_ratio)
    # test_end non serve esplicitamente: tutto quello che rimane va in test

    train_prefixes = pairs_prefixes[:train_end]
    val_prefixes   = pairs_prefixes[train_end:val_end]
    test_prefixes  = pairs_prefixes[val_end:]

    print(f"Totale coppie trovate: {num_total}")
    print(f" - Train: {len(train_prefixes)}")
    print(f" - Val:   {len(val_prefixes)}")
    print(f" - Test:  {len(test_prefixes)}")

    # 4) Funzione per copiare i file dati prefix e cartella di destinazione
    def copy_pair(prefix, subset_folder):
        lf_name = prefix + "_label_free" + extension
        st_name = prefix + "_stained" + extension

        lf_src = os.path.join(aligned_dir, lf_name)
        st_src = os.path.join(aligned_dir, st_name)

        lf_dst = os.path.join(output_dir, subset_folder, lf_name)
        st_dst = os.path.join(output_dir, subset_folder, st_name)

        shutil.copy2(lf_src, lf_dst)
        shutil.copy2(st_src, st_dst)

    # 5) Copia effettiva delle coppie
    for pfx in train_prefixes:
        copy_pair(pfx, "train")
    for pfx in val_prefixes:
        copy_pair(pfx, "val")
    for pfx in test_prefixes:
        copy_pair(pfx, "test")

    print("Suddivisione completata con successo!")


In [8]:
# ESEMPIO DI UTILIZZO:
if __name__ == "__main__":
    aligned_dir = "../../Materiale/Locale/aligned"       # cartella con tutte le 3000 coppie
    output_dir  = "../../Materiale/Locale/dataset_split" # cartella dove creare train/val/test

    split_dataset(aligned_dir, output_dir,
                  train_ratio=0.70, val_ratio=0.15, test_ratio=0.15,
                  extension=".tif", seed=123)

Totale file _label_free trovati: 3039
Totale coppie trovate: 3039
Elenco primi 50 prefissi: ['00000_09300', '00000_09600', '00000_09900', '00000_10200', '00000_10500', '00000_10800', '00000_11100', '00000_11400', '00000_11700', '00000_12000', '00300_08700', '00300_09000', '00300_09300', '00300_09600', '00300_09900', '00300_10200', '00300_10500', '00300_10800', '00300_11100', '00300_11400', '00300_11700', '00300_12000', '00300_12300', '00600_06300', '00600_06600', '00600_06900', '00600_07200', '00600_07500', '00600_07800', '00600_08100', '00600_08400', '00600_08700', '00600_09000', '00600_09300', '00600_09600', '00600_09900', '00600_10200', '00600_10500', '00600_10800', '00600_11100', '00600_11400', '00600_11700', '00600_12000', '00600_12300', '00600_15000', '00600_15300', '00900_05700', '00900_06000', '00900_06300', '00900_06600']
Totale coppie trovate: 3039
 - Train: 2127
 - Val:   455
 - Test:  457
Suddivisione completata con successo!


# Inizio Pytorch

Per verificare che Pytorch veda i Cuda cores della scheda video (in questo caso Nvidia RTX 3060Ti) eseguamo lo script.  

In [9]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))


PyTorch version: 2.5.1+cu121
CUDA available: True
Device: NVIDIA GeForce RTX 3060 Ti


Analizziamo ora un esempio di codice scritto con pytorch per distinguere gli elementi caratteristici dei nostri futuri algoritmi:  

## IMPORTANTE:
I seguenti pezzi di codice non andranno eseguiti da notebook ma direttamente da terminale poiché rischiano di creare problemi e non funzionare correttamente.  

In [ ]:
import os
import time
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

Di seguito è riportata una classe che raccoglie gli elementi principali del nostro dataset.  

Cosa fa il codice? Il codice prova a generare una serie di immagini colorate e salva sulla cartella la triade label-free, output, target (la stained)

La seguente dichiaraizone della classe `PairedHistologyDataset` ci permette di dichiarare una classe con quell'identificativo e di assegnare ad essa quattro funzioni privati: `__init__`, `__get_pairs__`, `__len__` e infine `__getitem__`.  
- `__init__`: costruttore;
- `__get_pairs__`: scorre tutti i file nella cartella e cerca i file che terminano con `label_free.tif`. Per ognuno di esso controlla che esista il corrispettivo stained con lo stesso prefisso e infine ritorna una lista di prefissi validi;
- `__len__`: resituisce la lunghezza della lista delle coppie valide;
- `__getitem__`: ricarica i path, apre le immagini tramite PIL e applica le trasformazioni successivamente spiegate;  

La classe `PairedHistologyDataset` eredita da Pytorch la classe `Dataset` la quale contiene **necessariamente** le funzioni `__len__` e `__getitem__`, quindi queste due devono essere sempre presenti e con gli stessi nomi. Questo passaggio è necessario affinchè possiamo creare un dataset sul quale lavorare con Pytorch.

In [ ]:
# -------------------------------------------------------
# Dataset personalizzato
# -------------------------------------------------------
class PairedHistologyDataset(Dataset):
    def __init__(self, folder_path, transform=None):
        self.folder_path = folder_path
        self.transform = transform
        self.pairs = self._get_pairs()

    def _get_pairs(self):
        """Cerca tutti i file che finiscono con '_label_free.tif'
           e costruisce la lista dei prefissi."""
        all_files = os.listdir(self.folder_path)
        prefixes = []
        for f in all_files:
            if f.endswith("_label_free.tif"):
                prefix = f.replace("_label_free.tif", "")
                # Controllo che esista anche '_stained.tif'
                stained_file = prefix + "_stained.tif"
                if stained_file in all_files:
                    prefixes.append(prefix)
        return sorted(prefixes)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        prefix = self.pairs[idx]
        lf_path = os.path.join(self.folder_path, prefix + "_label_free.tif")
        st_path = os.path.join(self.folder_path, prefix + "_stained.tif")

        # Carico entrambe le immagini
        lf_img = Image.open(lf_path).convert("RGB")
        st_img = Image.open(st_path).convert("RGB")

        # Applico le eventuali trasformazioni
        if self.transform:
            lf_img = self.transform(lf_img)
            st_img = self.transform(st_img)

        return lf_img, st_img

La trasformazione seguente applica un ridimensionamento dell'immagine per ottenere un 512x512 (che tra l'altro coincide con la dimensione delle immagini di partenza) e le converte in un tensore (un array `torch.Tensor` 3D \[canali, altezza, larghezza\] con valori tra 0 e 1).  
Le immagini caricate tramite PIL o OpenCV sono matrici intere con valori compresi tra 0 e 255 e 3 canali, solo che la rete esploderebbe con questi numero perciò la si normalizza e si passa dalla forma \[H, W, 3\] a \[3, H, W\] e ogni pixel P diviene P/255, quindi assumendo valori tra 0.0 e 1.0.  

In [ ]:
# -------------------------------------------------------
# Trasformazioni
# -------------------------------------------------------
transform = transforms.Compose([
    transforms.Resize((512, 512)),   # semplifica
    transforms.ToTensor(),           # converte in tensor [C,H,W] in [0,1]
])

Ora creiamo una semplice classe rappresentante il nostro modello Simple Convolutional Neural Network (`SimpleConvNet`).  
In esecuzione del costruttore eseguiamo anche `super().__init__()`, la quale ci permette di eseguire anche il costruttore della classe madre `nn.Module`, ovvero la classe base di tutte le reti in PyTorch. Senza queste ereditarietà tutto sarebbe molto più complesso.  
- `nn.Sequential` è un modo rapido per definire una rete come una sequenza di layer che vengono eseguiti uno dopo l'altro nell'ordine in cui sono scritti, quindi `self.net(x)` sarà come fare x → layer1 → layer2 → layer3 → layer4 → output
Quelli dichiarati all'interno sono i layer e sono:
1. `nn.Conv2d`: prende in input un'immagine e restituisce un nuovo volume di caratteristiche (feature map). E' definita come `nn.conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, ...)`.
    - L'input ha 3 canali (RGB)
    - Produce 32 immagini diverse, ognuna risultante dell'applicazione di un filtro (insieme di pesi variabile) diverso
    - Ogni filtro è calcolato su una dimensione quadrata del kernel pari a 3, ovvero un 3x3, percoò ogni layer guarda ai suoi vicini diretti (8 pixel attorno)
    - Con padding a 1 inserisco un bordo di un pixel assicurandomi che l'immagine dopo la convoluzione sarà ancora 512x512
    - Il parametro stride impostato a 1 di default indica lo spostamento del filtro
2. Dopo la convoluzione eseguita nello step precedente potremmo ottenere anche valori negativi, i quali verranno eliminati da Rectified Linear Unit (ReLU). Questo induce perdita di informazione ma al momento è trascurabile. **Alternative future potrebbero essere LeakyReLU, Tanh, ELU, GELU...**
3. Dopo la prima convoluzione il nostro output **teorico** sarà \[32, H, W\]. Alcuni di questi canali possono essere completamente a zero se ReLU ha annullato tutto, ma questo non comporta una rimozione di essi, semplicemente il neurone del layer successivo potrebbe ricevere uno zero. Questo output sarà il nostro input per una seconda convoluzione, sempre con `nn.Conv2d`, la quale ricombina le parti rilevanti dei 32 canali in ingresso per produrre un immagine RGB finale
4. `nn.Tanh` (tangente iperbolica) è l'ultimo layer, quello che produce l'immagine finale generata

In [ ]:
# -------------------------------------------------------
# Semplice modello di test
# -------------------------------------------------------
class SimpleConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 3, kernel_size=3, padding=1),
            nn.Tanh()  # output in [-1, 1]
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
def main():
    # -------------------------------------------------------
    # Creazione dataset e dataloader
    # -------------------------------------------------------
    train_folder = "Materiale/Locale/dataset_split/train" 
    train_dataset = PairedHistologyDataset(train_folder, transform=transform)

    # Imposta batch_size
    train_loader = DataLoader(
        train_dataset,
        batch_size=16,
        shuffle=True,
        num_workers=4,   # 0 => nessun worker parallelo (profiling più leggibile)
        pin_memory=True if torch.cuda.is_available() else False
    )

    # -------------------------------------------------------
    # Inizializzazione
    # -------------------------------------------------------
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device in uso:", device)

    model = SimpleConvNet().to(device)
    criterion = nn.L1Loss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    # -------------------------------------------------------
    # Training loop + mini-profiler con time.time()
    # -------------------------------------------------------
    print("Inizio training di prova...\n")
    model.train()

    # Crea la cartella per le immagini di preview
    os.makedirs("Materiale/Locale/output_preview", exist_ok=True)

    num_epochs = 2  # due epoche di test
    for epoch in range(num_epochs):
        running_loss = 0.0
        start_time = time.time()

        for i, (input_img, target_img) in enumerate(train_loader):
            
            # Trasferisci su GPU
            input_img = input_img.to(device, non_blocking=True)
            target_img = target_img.to(device, non_blocking=True)

            # Forward + backward
            output = model(input_img)
            loss = criterion(output, target_img)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            # Salva le immagini ogni 10 batch
            if i % 10 == 0:
                # output è nel range [-1, 1] → riportiamolo in [0, 1] per salvarlo
                output_vis = (output.detach().cpu() + 1) / 2.0
                target_vis = (target_img.detach().cpu() + 1) / 2.0
                input_vis  = input_img.detach().cpu()  # già in [0, 1]

                # Salviamo solo la prima immagine del batch
                vutils.save_image(input_vis[0], f"output_preview/ep{epoch+1:02d}_b{i:03d}_input.png")
                vutils.save_image(output_vis[0], f"output_preview/ep{epoch+1:02d}_b{i:03d}_output.png")
                vutils.save_image(target_vis[0], f"output_preview/ep{epoch+1:02d}_b{i:03d}_target.png")

                print(f"    Salvata preview batch {i} in 'output_preview/'")

        # fine epoca
        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs} - Loss media: {avg_loss:.4f}")

    print("Training completato!")

In [ ]:
if __name__ == "__main__":
    main()